# 5. GPT-2 inference with metrics


In [1]:
# Dynamicly load evaluation metrics and the model
%run -i ../src/models/evaluation.py
%run -i ../src/models/detoxGPT2.py

In [2]:
import numpy as np
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
detoxGPT = detoxGPT2("../models/gpt2-based")

In [4]:
prompt = "I hate your stupid fucking gugly face!"

In [5]:
# Pack the suggestions into a dataframe
df = pd.DataFrame(
    detoxGPT.get_detoxed_suggestions(prompt, device=DEVICE), columns=["suggestion"]
)

# Add empty column for each metric
metrics = ["wo", "cs", "bleu"]
df[metrics] = pd.DataFrame([[0] * len(metrics)], index=df.index, dtype=float)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
# Instantiate metric classes
similarity = Similarity()
toxicity = STAToxic()

In [7]:
# Generate toxicity report for each suggestion
toxicity_report = toxicity.toxicity_report(df["suggestion"])
toxicity_report_columns = toxicity_report.columns

for index, row in df.iterrows():
    df.loc[index, "wo"] = similarity.get_wo_score(prompt, row["suggestion"])
    df.loc[index, "cs"] = similarity.get_cosine_score(prompt, row["suggestion"])
    df.loc[index, "bleu"] = similarity.get_bleu_score(prompt, row["suggestion"])

# Concat with toxicity report
df = pd.concat([df, toxicity_report], axis=1)
df

c:\Users\danielpancake\Desktop\text-detox\.venv\Lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\danielpancake\Desktop\text-detox\.venv\Lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,I hate your face!,0.375000,0.984766,2.749736e-01,0.896557,0.061447,0.139865,0.086860,0.281614,0.048960
1,I hate that ugly face!,0.200000,0.938467,3.354195e-01,0.988258,0.111846,0.191352,0.130032,0.717605,0.110246
2,I hate the ugly face!,0.200000,0.925100,3.248289e-01,0.988258,0.111846,0.191352,0.130032,0.717605,0.110246
3,I hate that ugly face.,0.200000,0.894888,3.113067e-01,0.988258,0.111846,0.191352,0.130032,0.717605,0.110246
4,I hate your face!,0.375000,0.984766,2.749736e-01,0.896557,0.061447,0.139865,0.086860,0.281614,0.048960
5,"Oh, I hate your face",0.500000,0.894655,3.042480e-01,0.913597,0.060312,0.243640,0.025296,0.294555,0.092565
6,I hate you too.,0.222222,0.893378,1.414649e-01,0.874505,0.037967,0.065795,0.005758,0.158340,0.070589
7,I don't like that messy face,0.181818,0.862821,1.972656e-01,0.438473,0.004851,0.042671,0.017338,0.102742,0.007704
8,I hate my nose in!,0.200000,0.835429,1.422419e-01,0.861140,0.009859,0.101126,0.001426,0.031359,0.057165
9,I hated the angry face!,0.090909,0.862190,2.796013e-01,0.506654,0.075687,0.086214,0.047256,0.082433,0.002293


In [8]:
# Calculate the score
# Toxicity report should be as low as possible
# Similarity metrics should be as high as possible (use geometric mean)
df["score"] = 1 / df[toxicity_report_columns].sum(axis=1)
df["score"] *= np.power(df[metrics].prod(axis=1), 1 / len(metrics))
df = df.sort_values(by=["score"], ascending=False)
df

,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate,score
7,I don't like that messy face,0.181818,0.862821,1.972656e-01,0.438473,0.004851,0.042671,0.017338,0.102742,0.007704,5.115156e-01
9,I hated the angry face!,0.090909,0.862190,2.796013e-01,0.506654,0.075687,0.086214,0.047256,0.082433,0.002293,3.495705e-01
5,"Oh, I hate your face",0.500000,0.894655,3.042480e-01,0.913597,0.060312,0.243640,0.025296,0.294555,0.092565,3.155777e-01
0,I hate your face!,0.375000,0.984766,2.749736e-01,0.896557,0.061447,0.139865,0.086860,0.281614,0.048960,3.078832e-01
4,I hate your face!,0.375000,0.984766,2.749736e-01,0.896557,0.061447,0.139865,0.086860,0.281614,0.048960,3.078832e-01
8,I hate my nose in!,0.200000,0.835429,1.422419e-01,0.861140,0.009859,0.101126,0.001426,0.031359,0.057165,2.707074e-01
6,I hate you too.,0.222222,0.893378,1.414649e-01,0.874505,0.037967,0.065795,0.005758,0.158340,0.070589,2.505992e-01
1,I hate that ugly face!,0.200000,0.938467,3.354195e-01,0.988258,0.111846,0.191352,0.130032,0.717605,0.110246,1.768577e-01
2,I hate the ugly face!,0.200000,0.925100,3.248289e-01,0.988258,0.111846,0.191352,0.130032,0.717605,0.110246,1.741417e-01
3,I hate that ugly face.,0.200000,0.894888,3.113067e-01,0.988258,0.111846,0.191352,0.130032,0.717605,0.110246,1.698012e-01


In [10]:
# Pick the suggestion with the highest score
suggestion = df.iloc[0]["suggestion"]
print(f"{prompt} -> {suggestion}")

I hate your stupid fucking gugly face! -> I don't like that messy face
